# Combined Power-Law Plot

This notebook combines the first two power-law panels with the first layer-wise centered panel into one `(1, 3)` figure.


In [ ]:
import os
import glob

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


def set_publication_style():
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
        'mathtext.fontset': 'stix',
        'font.size': 18,
        'axes.labelsize': 22,
        'axes.titlesize': 18,
        'xtick.labelsize': 20,
        'ytick.labelsize': 20,
        'legend.fontsize': 11,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'axes.linewidth': 1.6,
        'lines.linewidth': 1.4,
        'xtick.major.width': 1.6,
        'ytick.major.width': 1.6,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
    })


set_publication_style()


In [ ]:
def extract_diagonals(A, B, slice_range=None):
    if slice_range is None:
        slice_range = torch.arange(min(A.shape[0], B.shape[0]))
    A = torch.flip(A, dims=[0, 1])
    B = torch.flip(B, dims=[0, 1])
    A_dense = torch.diag_embed(torch.diag(A)[slice_range])
    B_dense = torch.diag_embed(torch.diag(B)[slice_range])
    return A_dense, B_dense


def compute_centered_loglog_fit(A_dense, B_dense):
    diag_a = A_dense.diag().detach().cpu().numpy()
    diag_b = B_dense.diag().detach().cpu().numpy()

    mask = (diag_a > 0) & (diag_b > 0)
    if mask.sum() < 10:
        return None

    log_a = np.log10(diag_a[mask])
    log_b = np.log10(diag_b[mask])
    slope, intercept, r_value, _, _ = stats.linregress(log_b, log_a)
    return {
        'x': log_b - log_b.mean(),
        'y': log_a - log_a.mean(),
        'slope': float(slope),
        'intercept': float(intercept),
        'r2': float(r_value ** 2),
    }


def resolve_first_group_dir(config):
    train_size = config['train_size']
    sample_number = config['sample_number']
    class_num = config['class_number']
    return (
        f"./AWCH_data/TrainSize{train_size}_SampleN{sample_number}_"
        f"ClassN{class_num}_B{config['B']}lr{config['alpha']}_"
        f"lossfn_{config['lss_fn']}_model_{config['model']}_dataset_{config['dataset']}"
    )


def load_first_group_results(config, epoch, slice_range, var_pairs):
    save_dir = resolve_first_group_dir(config)
    file_path = os.path.join(save_dir, f"C_epoch_{epoch}.pt")
    if not os.path.exists(file_path):
        print(f"[Missing] {config['label']} -> {file_path}")
        return {}

    try:
        loaded_data = torch.load(file_path, map_location='cpu')
    except Exception as exc:
        print(f"[Error] Loading {config['label']}: {exc}")
        return {}

    if 'Hessian' not in loaded_data:
        print(f"[MissingKey] {config['label']} has no Hessian key.")
        return {}

    try:
        _, V = torch.linalg.eigh(torch.tensor(loaded_data['Hessian']).double())
        V = V.float()
    except Exception as exc:
        print(f"[Error] Eigen decomposition failed for {config['label']}: {exc}")
        return {}

    processed = {}
    for pair_name, (key_a, key_b) in var_pairs.items():
        if key_a not in loaded_data or key_b not in loaded_data:
            print(f"[SkipPair] {config['label']} missing {key_a} or {key_b}")
            continue

        A = torch.tensor(loaded_data[key_a]).float()
        B = torch.tensor(loaded_data[key_b]).float()

        if any(sub in key_a.lower() for sub in ['covar', 'hessian']):
            A = V.transpose(-2, -1) @ A @ V
        if any(sub in key_b.lower() for sub in ['covar', 'hessian']):
            B = V.transpose(-2, -1) @ B @ V

        A_dense, B_dense = extract_diagonals(A, B, slice_range)
        fit_data = compute_centered_loglog_fit(A_dense, B_dense)
        if fit_data is None:
            print(f"[SkipPair] {config['label']} {pair_name}: insufficient positive diagonal entries")
            continue

        fit_data.update({
            'label': config['label'],
            'color': config.get('color', 'black'),
        })
        processed[pair_name] = fit_data

    return processed


In [ ]:
def extract_diagonals_layerwise(A, B, slice_range=None):
    n = min(A.shape[0], B.shape[0])
    diag_a = torch.diag(A)[:n]
    diag_b = torch.diag(B)[:n]

    sorted_idx = torch.argsort(diag_b, descending=True)
    diag_a = diag_a[sorted_idx]
    diag_b = diag_b[sorted_idx]

    if slice_range is not None:
        idx = slice_range[slice_range < n]
        if idx.numel() == 0:
            idx = torch.arange(n)
        diag_a = diag_a[idx]
        diag_b = diag_b[idx]

    return torch.diag_embed(diag_a), torch.diag_embed(diag_b)


def resolve_save_dir_for_layer(cfg, layer_index):
    train_size = cfg['train_size']
    sample_number = cfg['sample_number']
    class_num = cfg['class_number']
    hidden_sizes = cfg.get('hidden_sizes', [50, 40, 30, 20])
    hidden_sizes_compact = str(hidden_sizes).replace(' ', '')

    exact_candidates = [
        f"./AWCH_data/HS{hidden_sizes}_layer{layer_index}_TrainSize{train_size}_SampleN{sample_number}_ClassN{class_num}_B{cfg['B']}lr{cfg['alpha']}_lossfn_{cfg['lss_fn']}_model_{cfg['model']}_dataset_{cfg['dataset']}",
        f"./AWCH_data/HS{hidden_sizes_compact}_layer{layer_index}_TrainSize{train_size}_SampleN{sample_number}_ClassN{class_num}_B{cfg['B']}lr{cfg['alpha']}_lossfn_{cfg['lss_fn']}_model_{cfg['model']}_dataset_{cfg['dataset']}",
    ]
    existing_exact = [candidate for candidate in exact_candidates if os.path.isdir(candidate)]
    if len(existing_exact) == 1:
        return existing_exact[0]
    if len(existing_exact) > 1:
        print(f"[Ambiguous] Layer {layer_index}: multiple exact directories found, using first: {existing_exact[0]}")
        return existing_exact[0]

    print(f"[Missing] Layer {layer_index}: exact directory not found.")
    for candidate in exact_candidates:
        print(f"  - {candidate}")
    return None


def collect_layer_results_with_prefactor(
    base_cfg,
    layer_indices,
    epoch_to_plot,
    slice_range,
    var_pairs_to_plot,
    palette,
    fit_fraction=0.8,
):
    results = []
    for i, layer_idx in enumerate(layer_indices):
        cfg = dict(base_cfg)
        cfg['label'] = f"Layer {layer_idx}"
        cfg['color'] = palette[i % len(palette)]

        save_dir = resolve_save_dir_for_layer(cfg, layer_idx)
        if save_dir is None:
            results.append({})
            continue

        file_path = os.path.join(save_dir, f"C_epoch_{epoch_to_plot}.pt")
        if not os.path.exists(file_path):
            print(f"[Missing] Layer {layer_idx}: {file_path}")
            results.append({})
            continue

        print(f"[Layer {layer_idx}] using file: {file_path}")
        try:
            loaded_data = torch.load(file_path, map_location='cpu')
            _, V = torch.linalg.eigh(torch.tensor(loaded_data['Hessian']).double())
            V = V.float()
        except Exception as exc:
            print(f"[Error] Layer {layer_idx} loading/eigh failed: {exc}")
            results.append({})
            continue

        one_layer = {}
        for pair_name, (key_a, key_b) in var_pairs_to_plot.items():
            if key_a not in loaded_data or key_b not in loaded_data:
                print(f"[SkipPair] Layer {layer_idx} missing {key_a} or {key_b}")
                continue

            A = torch.tensor(loaded_data[key_a]).float()
            B = torch.tensor(loaded_data[key_b]).float()

            if 'covar' in key_a.lower() or 'hessian' in key_a.lower():
                A = V.transpose(-2, -1) @ A @ V
            if 'covar' in key_b.lower() or 'hessian' in key_b.lower():
                B = V.transpose(-2, -1) @ B @ V

            A_dense, B_dense = extract_diagonals_layerwise(A, B, slice_range)
            diag_a = A_dense.diag().detach().cpu().numpy()
            diag_b = B_dense.diag().detach().cpu().numpy()
            mask = (diag_a > 0) & (diag_b > 0)
            if mask.sum() < 10:
                continue

            log_a_raw = np.log10(diag_a[mask])
            log_b_raw = np.log10(diag_b[mask])

            keep = (
                (log_a_raw >= np.percentile(log_a_raw, 5))
                & (log_a_raw <= np.percentile(log_a_raw, 100))
                & (log_b_raw >= np.percentile(log_b_raw, 5))
                & (log_b_raw <= np.percentile(log_b_raw, 100))
            )
            log_a_raw = log_a_raw[keep]
            log_b_raw = log_b_raw[keep]
            if log_a_raw.size < 10:
                continue

            if fit_fraction <= 0:
                continue
            if 0 < fit_fraction < 1 and log_a_raw.size > 2:
                n_fit = max(2, int(np.ceil(log_a_raw.size * fit_fraction)))
                log_a_fit = log_a_raw[:n_fit]
                log_b_fit = log_b_raw[:n_fit]
            else:
                log_a_fit = log_a_raw
                log_b_fit = log_b_raw

            slope, intercept, r_value, _, _ = stats.linregress(log_b_fit, log_a_fit)
            one_layer[pair_name] = {
                'x_raw': log_b_raw,
                'y_raw': log_a_raw,
                'slope': float(slope),
                'intercept': float(intercept),
                'prefactor': float(10.0 ** intercept),
                'r2': float(r_value ** 2),
                'label': cfg['label'],
                'color': cfg['color'],
                'fit_points': int(log_a_fit.size),
                'total_points': int(log_a_raw.size),
            }

        results.append(one_layer)

    return results


In [ ]:
def fit_legend_label(label, slope, r2):
    return f"{label} (slope={slope:.2f}, R^2={r2:.3f})"


def add_reference_lines(ax, x_values, y_values, line_width=1.2):
    min_x, max_x = float(np.min(x_values)), float(np.max(x_values))
    min_y, max_y = float(np.min(y_values)), float(np.max(y_values))
    cx = 0.5 * (min_x + max_x)
    cy = 0.5 * (min_y + max_y)
    ref_x = np.array([min_x, max_x])

    ax.plot(ref_x, 1.0 * (ref_x - cx) + cy, 'k--', lw=line_width, alpha=0.9, label='Slope=1 (Ref)')
    ax.plot(ref_x, 2.0 * (ref_x - cx) + cy, 'k:', lw=line_width, alpha=0.9, label='Slope=2 (Ref)')

    x_half = max(0.5, 0.5 * (max_x - min_x)) * 1.15
    y_half = max(0.5, 0.5 * (max_y - min_y)) * 1.15
    ax.set_xlim(cx - x_half, cx + x_half)
    ax.set_ylim(cy - y_half, cy + y_half)


def style_power_law_axis(ax, title, xlabel, ylabel=None, legend_loc='upper left'):
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.grid(True, linestyle='--', alpha=0.8)
    ax.legend(loc=legend_loc, frameon=True, framealpha=0.9, fontsize=11)
    ax.set_aspect('equal', adjustable='datalim')


def plot_first_group_panel(ax, results_list, pair_name, key_a, key_b, y_shift_step=1.5):
    all_x = []
    all_y = []

    for config_idx, res_dict in enumerate(results_list):
        if pair_name not in res_dict:
            continue

        data = res_dict[pair_name]
        x = data['x']
        y = data['y']
        current_shift = config_idx * y_shift_step
        y_shifted = y - current_shift

        ax.scatter(x, y_shifted, s=8, alpha=0.28, color=data['color'], edgecolors='none')
        x_fit = np.array([x.min(), x.max()])
        y_fit = data['slope'] * x_fit - current_shift
        ax.plot(
            x_fit,
            y_fit,
            '-',
            lw=1.2,
            color=data['color'],
            label=fit_legend_label(data['label'], data['slope'], data['r2']),
        )

        all_x.extend([x.min(), x.max()])
        all_y.extend([y_shifted.min(), y_shifted.max()])

    if all_x:
        add_reference_lines(ax, np.array(all_x), np.array(all_y), line_width=1.2)
    else:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')

    if 'covar' in key_a.lower():
        title = 'Empirical Covariance'
    elif key_a == 'H_2_d':
        title = 'AWD-derived Covariance'
    else:
        title = pair_name

    style_power_law_axis(
        ax,
        title=title,
        xlabel=f"Centered $\\log_{{10}}$({key_b})",
        ylabel=f"Shifted $\\log_{{10}}$({key_a})" if key_a == 'Covar' else None,
        legend_loc='upper left',
    )


def plot_layerwise_centered_panel(
    ax,
    layer_results,
    pair_name,
    key_a,
    key_b,
    base_cfg,
    y_shift_step=2.5,
    plot_fraction=0.8,
):
    series = []
    raw_x_all = []
    raw_y_all = []

    for layer_pos, layer_dict in enumerate(layer_results):
        if pair_name not in layer_dict:
            continue
        data = layer_dict[pair_name]
        series.append((layer_pos, data))
        raw_x_all.append(data['x_raw'])
        raw_y_all.append(data['y_raw'])

    if not series:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
        style_power_law_axis(
            ax,
            title='Covar vs Hessian (Centered + Shift)',
            xlabel=f"Centered $\\log_{{10}}$({key_b})",
            ylabel=None,
            legend_loc='upper left',
        )
        return

    center_x = float(np.median(np.concatenate(raw_x_all)))
    center_y = float(np.median(np.concatenate(raw_y_all)))
    x_view_all = []
    y_view_all = []

    for layer_pos, data in series:
        y_shift = layer_pos * y_shift_step
        x_plot = data['x_raw'] - center_x
        y_plot = data['y_raw'] - center_y - y_shift

        if 0 < plot_fraction < 1 and x_plot.size > 1:
            keep_n = max(1, int(np.ceil(x_plot.size * plot_fraction)))
            x_vis = x_plot[:keep_n]
            y_vis = y_plot[:keep_n]
        elif plot_fraction <= 0:
            x_vis = x_plot[:1]
            y_vis = y_plot[:1]
        else:
            x_vis = x_plot
            y_vis = y_plot

        if x_vis.size == 0:
            continue

        x_view_all.append(x_vis)
        y_view_all.append(y_vis)
        ax.scatter(x_vis, y_vis, s=12, alpha=0.35, color=data['color'], edgecolors='none')

        if x_vis.size >= 2:
            x_line = np.array([0.8 * x_vis.min(), 1.1 * x_vis.max()])
        else:
            x_line = np.array([0.8 * x_plot.min(), 1.1 * x_plot.max()])
        x_line_raw = x_line + center_x
        y_line_raw = data['slope'] * x_line_raw + data['intercept']
        y_line = y_line_raw - center_y - y_shift
        ax.plot(
            x_line,
            y_line,
            '-',
            lw=1.2,
            color=data['color'],
            label=fit_legend_label(data['label'], data['slope'], data['r2']),
        )

    if x_view_all:
        add_reference_lines(ax, np.concatenate(x_view_all), np.concatenate(y_view_all), line_width=1.2)

    display_model = 'MLP_multilayer' if base_cfg['model'] == 'FC_multilayer' else base_cfg['model']
    display_dataset = base_cfg['dataset'].upper()
    display_loss = 'CE' if base_cfg['lss_fn'] == 'cse' else base_cfg['lss_fn'].upper()
    model_title = f"{display_model} / {display_dataset} / {display_loss}, hidden_sizes={base_cfg['hidden_sizes']}"
    style_power_law_axis(
        ax,
        title=f"Covar vs Hessian (Centered + Shift)\n{model_title}",
        xlabel=f"Centered $\\log_{{10}}$({key_b})",
        ylabel=None,
        legend_loc='upper left',
    )


def plot_combined_power_law_1x3(
    first_results,
    first_var_pairs,
    layer_results,
    layer_pair_name,
    layer_pair_keys,
    base_cfg,
    first_y_shift_step=1.5,
    layer_y_shift_step=2.5,
    layer_plot_fraction=0.8,
    filename='power_law_combined_1x3.pdf',
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    first_pairs = list(first_var_pairs.items())
    for ax, (pair_name, (key_a, key_b)) in zip(axes[:2], first_pairs[:2]):
        plot_first_group_panel(
            ax,
            first_results,
            pair_name,
            key_a,
            key_b,
            y_shift_step=first_y_shift_step,
        )

    plot_layerwise_centered_panel(
        axes[2],
        layer_results,
        layer_pair_name,
        layer_pair_keys[0],
        layer_pair_keys[1],
        base_cfg,
        y_shift_step=layer_y_shift_step,
        plot_fraction=layer_plot_fraction,
    )

    plt.tight_layout()
    os.makedirs('ICML_Figures', exist_ok=True)
    save_path = os.path.join('ICML_Figures', filename)
    plt.savefig(save_path, format='pdf', bbox_inches='tight', pad_inches=0.05, dpi=300)
    print(f"Saved figure to: {save_path}")
    plt.show()
    return save_path


In [ ]:
# First two panels: current parameters from the original 1x2 log-log plot.
epoch_to_plot = 100
slice_range_first = torch.arange(0, 1000)

first_var_pairs = {
    'Covar vs Hessian': ('Covar', 'Hessian'),
    'H_2 vs Hessian': ('H_2_d', 'Hessian'),
}

first_configs = [
    {
        'label': 'CNN/CIFAR10/CE',
        'lss_fn': 'cse',
        'dataset': 'cifar10',
        'model': 'CNN',
        'B': 128,
        'alpha': 0.1,
        'train_size': 2000,
        'sample_number': 20,
        'class_number': 10,
        'color': '#1f77b4',
    },
    {
        'label': 'MLP/MNIST/CE',
        'lss_fn': 'cse',
        'dataset': 'mnist',
        'model': 'FC',
        'B': 50,
        'alpha': 0.1,
        'train_size': 2000,
        'sample_number': 20,
        'class_number': 10,
        'color': '#d62728',
    },
    {
        'label': 'MLP/MNIST/MSE',
        'lss_fn': 'mse',
        'dataset': 'mnist',
        'model': 'FC',
        'B': 50,
        'alpha': 0.1,
        'train_size': 2000,
        'sample_number': 20,
        'class_number': 10,
        'color': '#2ca02c',
    },
]

# Third panel: current parameters from the following layer-wise code.
base_cfg = {
    'lss_fn': 'cse',
    'dataset': 'mnist',
    'model': 'FC_multilayer',
    'B': 50,
    'alpha': 0.1,
    'train_size': 2000,
    'sample_number': 20,
    'class_number': 10,
    'hidden_sizes': [50, 50, 50, 50],
}
layer_indices = [1, 2, 3, 4]
layer_palette = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e']
layer_var_pairs = {
    'Covar vs Hessian': ('Covar', 'Hessian'),
    'H_2 vs Hessian': ('H_2_d', 'Hessian'),
}

fit_fraction = 0.8
plot_fraction = 0.8
first_y_shift_step = 1.5
layer_y_shift_step = 2.5


In [ ]:
print(f"--- Loading first two panels for epoch {epoch_to_plot} ---")
first_results = []
for cfg in first_configs:
    print(f"Processing: {cfg['label']}")
    first_results.append(load_first_group_results(cfg, epoch_to_plot, slice_range_first, first_var_pairs))

print('--- Loading third layer-wise panel ---')
layer_results = collect_layer_results_with_prefactor(
    base_cfg,
    layer_indices,
    epoch_to_plot,
    slice_range=None,
    var_pairs_to_plot=layer_var_pairs,
    palette=layer_palette,
    fit_fraction=fit_fraction,
)

if not any(first_results):
    raise RuntimeError('No data loaded for the first two panels.')
if not any(layer_results):
    raise RuntimeError('No data loaded for the layer-wise third panel.')

combined_figure_path = plot_combined_power_law_1x3(
    first_results=first_results,
    first_var_pairs=first_var_pairs,
    layer_results=layer_results,
    layer_pair_name='Covar vs Hessian',
    layer_pair_keys=layer_var_pairs['Covar vs Hessian'],
    base_cfg=base_cfg,
    first_y_shift_step=first_y_shift_step,
    layer_y_shift_step=layer_y_shift_step,
    layer_plot_fraction=plot_fraction,
    filename='power_law_combined_1x3.pdf',
)
combined_figure_path
